<a href="https://colab.research.google.com/github/sumair789-lgtm/Code-switching-codesaviours-si26--Sumair-/blob/main/SI26_Week7_Sumair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This is my Week 7 Notebook**
In this task , I re-uploaded myb dataset and train a model on it
After that I check the evaluation process

Google Drive Mounted

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Uploaded Dataset

In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(3035, 3)
['sentence', 'word', 'label']


,sentence,word,label
0,Aaj ka din bohot busy tha,Aaj,URD
1,Aaj ka din bohot busy tha,ka,URD
2,Aaj ka din bohot busy tha,din,URD
3,Aaj ka din bohot busy tha,bohot,URD
4,Aaj ka din bohot busy tha,busy,ENG


Model

In [4]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch datasets seqeval

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/dataset.csv')
print(f"GPU available: {torch.cuda.is_available()}")

print(df['label'].value_counts())
print(f"Total unique sentences: {df['sentence'].nunique()}")

label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

sentences = df.groupby('sentence').apply(
    lambda x: {'words': x['word'].tolist(), 'labels': x['label'].tolist()}
).tolist()

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4357b598f209f231f5149ee8d11d2b5be80661b26f0f2f9285b78e04b740e1cc
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
GPU available: True
label
URD    1697
ENG    1305
MIX      33
Name: count, dtype: int64
Total unique sentences: 379
Training sentences: 303
Testing sentences: 76


/tmp/ipykernel_647/16225075.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence').apply(


Checking MIx Labels

In [6]:
mix_sentences = [s for s in sentences if 'MIX' in s['labels']]
non_mix_sentences = [s for s in sentences if 'MIX' not in s['labels']]
print(f"MIX-containing sentences: {len(mix_sentences)} / {len(sentences)}")

if len(mix_sentences) >= 3:
    mix_train, mix_test = train_test_split(mix_sentences, test_size=0.3, random_state=42)
else:
    mix_train, mix_test = mix_sentences, mix_sentences[:1]  # kam hain to kam se kam 1 test mein force karo

non_mix_train, non_mix_test = train_test_split(non_mix_sentences, test_size=0.2, random_state=42)

train_data = mix_train + non_mix_train
test_data = mix_test + non_mix_test
print(f'Training sentences: {len(train_data)}, Testing sentences: {len(test_data)}, MIX in test: {len(mix_test)}')

MIX-containing sentences: 33 / 379
Training sentences: 299, Testing sentences: 80, MIX in test: 10


**Trainig Model**

In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    report_to='none',   # wandb login prompt se bachne ke liye
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

print('Starting training...')
trainer.train()
print('Training complete!')

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/299 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [8]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

./results/checkpoint-114
0.07565285265445709


**Evaluation**

In [9]:
import numpy as np
from sklearn.metrics import classification_report

predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=2)
true_labels = predictions.label_ids

true_flat, pred_flat = [], []
for true_seq, pred_seq in zip(true_labels, preds):
    for t, p in zip(true_seq, pred_seq):
        if t != -100:
            true_flat.append(id2label[t])
            pred_flat.append(id2label[p])

report = classification_report(
    true_flat, pred_flat,
    labels=['URD', 'ENG', 'MIX'],
    target_names=['URD', 'ENG', 'MIX'],
    zero_division=0
)
print(report)

              precision    recall  f1-score   support

         URD       1.00      0.98      0.99       402
         ENG       0.96      1.00      0.98       242
         MIX       0.89      0.80      0.84        10

    accuracy                           0.98       654
   macro avg       0.95      0.93      0.94       654
weighted avg       0.98      0.98      0.98       654



**Save and Push to hugging Face**

In [10]:
from huggingface_hub import notebook_login

# This will pop up an input box. Paste your HF Write Token here and press Enter.
notebook_login()

# Update repository name with your name
repo_name = 'code-switching-codesaviours-si26-sumair'

# Push model and tokenizer
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model successfully published!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hczjzgz/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpk8u0x9ia/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model successfully published!
